<a href="https://colab.research.google.com/github/Naufallm/RAG-Quantitative-Research-Engine-for-IDX/blob/main/DSC_Project_IDX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Member
- Ahmad Naufal Luthfan Marzuqi
- Syahrial Nur Faturrahman

# Step 1: Install Library
Pada tahap ini, kita menginstal library `pypdf` untuk membaca file PDF dan `langchain` untuk memproses teks.
Library `pandas` digunakan untuk manajemen data tabel jika diperlukan.

Pada Step ini akan diminta untuk restart runtime, setelah restart dapat dijalankan kembali mulai dari awal ya!


In [ ]:
!pip install -U pypdf langchain langchain-community langchain-text-splitters pandas

# Step 2: Sinkronisasi Data
Kita mengambil dataset laporan keuangan yang sudah dikumpulkan di GitHub agar bisa diproses langsung di environment Colab ini.


Untuk link github nya https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX

In [1]:
!git clone https://github.com/Naufallm/RAG-Quantitative-Research-Engine-for-IDX.git

Cloning into 'RAG-Quantitative-Research-Engine-for-IDX'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 112 (delta 0), reused 0 (delta 0), pack-reused 110 (from 3)
Receiving objects: 100% (112/112), 81.09 MiB | 16.33 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (59/59), done.


# Step 3: Verifikasi File PDF
Tahap ini memastikan bahwa folder `dataset_idx` telah terunduh dengan benar dan menghitung jumlah file yang siap diproses.

In [ ]:
import os

dataset_path = "/content/RAG-Quantitative-Research-Engine-for-IDX/dataset_idx"
pdf_files = [f for f in os.listdir(dataset_path) if f.endswith(".pdf")]
print("Jumlah file PDF:", len(pdf_files))
print("\nDaftar file PDF: ")
for file in pdf_files:
  print(file)

Jumlah file PDF: 50

Daftar file PDF: 
FinancialStatement-2026-I-RISE.pdf
FinancialStatement-2026-I-ARTO.pdf
FinancialStatement-2026-I-BBCA.pdf
FinancialStatement-2026-I-ESIP.pdf
FinancialStatement-2026-I-SONA.pdf
FinancialStatement-2026-I-SBMA.pdf
FinancialStatement-2026-I-IBFN.pdf
FinancialStatement-2026-I-GIAA.pdf
FinancialStatement-2026-I-EAST.pdf
FinancialStatement-2026-I-AUTO.pdf
FinancialStatement-2026-I-FORE.pdf
FinancialStatement-2026-I-BLOG.pdf
FinancialStatement-2026-I-BMRI.pdf
FinancialStatement-2026-I-PGAS.pdf
FinancialStatement-2026-I-DSNG.pdf
FinancialStatement-2026-I-ISSP.pdf
FinancialStatement-2026-I-MORA.pdf
FinancialStatement-2026-I-BNLI.pdf
FinancialStatement-2026-I-PMJS.pdf
FinancialStatement-2026-I-EDGE.pdf
FinancialStatement-2026-I-DCII.pdf
FinancialStatement-2026-I-MYTX.pdf
FinancialStatement-2026-I-BOLT.pdf
FinancialStatement-2026-I-PGLI.pdf
FinancialStatement-2026-I-IKPM.pdf
FinancialStatement-2026-I-CFIN.pdf
FinancialStatement-2026-I-MCOL.pdf
FinancialStateme

# Step 4: PDF Text Extraction
Di sini kita membaca setiap halaman PDF dan mengubahnya menjadi teks string.
Kita juga mengambil Nama Perusahaan (Ticker) dari nama file menggunakan fungsi split teks.

In [ ]:
from pypdf import PdfReader
from tqdm import tqdm
import os

all_raw_documents = []

total_files = len(pdf_files)

for filename in tqdm(pdf_files, desc="Memproses PDF", unit="file"):
    path = os.path.join(dataset_path, filename)

    try:
        reader = PdfReader(path)
        full_text = ""

        for page in reader.pages:
            page_content = page.extract_text()
            if page_content:
                full_text += page_content + "\n"

        ticker_name = filename.split("-")[-1].replace(".pdf", "")

        all_raw_documents.append({
            "ticker": ticker_name,
            "filename": filename,
            "text": full_text
        })

    except Exception as e:
        print(f"Gagal membaca {filename}: {e}")

print(f"Proses selesai. Berhasil mengekstrak {len(all_raw_documents)} dokumen.")

Memproses PDF: 100%|██████████| 50/50 [04:03<00:00,  4.88s/file]

Proses selesai. Berhasil mengekstrak 50 dokumen.


# Step 5: Hierarchical Chunking
Kita menggunakan paket 'langchain_text_splitters' yang merupakan standar terbaru dari LangChain.
Teks laporan keuangan akan dipecah menjadi bagian-bagian kecil agar dapat diproses secara efisien
oleh model bahasa (Groq/LLM).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import json

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

final_chunks = []

for doc in tqdm(all_raw_documents, desc="Membuat chunks", unit="dokumen"):
    chunks = text_splitter.split_text(doc["text"])

    for chunk in chunks:
        final_chunks.append({
            "ticker": doc["ticker"],
            "source": doc["filename"],
            "content": chunk
        })

print(f"Total chunks yang dihasilkan: {len(final_chunks)}")

# Simpan hasil chunks ke file lokal
with open("final_chunks.json", "w", encoding="utf-8") as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=2)

print("Chunks berhasil disimpan ke final_chunks.json")

Membuat chunks: 100%|██████████| 50/50 [00:00<00:00, 132.71dokumen/s]


Total chunks yang dihasilkan: 9744
Chunks berhasil disimpan ke final_chunks.json


# Step 6: Quality Control & Sampel Output
Tahap terakhir adalah memastikan data hasil potongan (chunk) memiliki metadata yang benar
dan menampilkan sampel isi datanya untuk bahan laporan checkpoint.

In [ ]:
print(f"Total File Berhasil Diproses : {len(all_raw_documents)}")
print(f"Total Potongan Teks (Chunks) : {len(final_chunks)}")
print("-" * 40)

if len(final_chunks) > 0:
    print(f"SAMPEL DATA DARI EMITEN     : {final_chunks[0]['ticker']}")
    print(f"SUMBER FILE ASLI            : {final_chunks[0]['source']}")
    print("-" * 40)
    print("CUPLIKAN ISI TEKS:")
    print(final_chunks[0]['content'][:600] + "...")

Total File Berhasil Diproses : 50
Total Potongan Teks (Chunks) : 9744
----------------------------------------
SAMPEL DATA DARI EMITEN     : RISE
SUMBER FILE ASLI            : FinancialStatement-2026-I-RISE.pdf
----------------------------------------
CUPLIKAN ISI TEKS:
Perseroan dengan ini menyampaikan laporan keuangan untuk periode 3 Bulan yang berakhir pada 31/03/2026 dengan ikhtisar sebagai berikut : 
  
Informasi mengenai anak perusahaan Perseroan sebagai berikut : 
  
  
Nomor Surat 104/SKE/CS/JSMS/IV/2026
Nama Emiten PT Jaya Sukses Makmur Sentosa Tbk.
Kode Emiten RISE
Perihal Penyampaian Laporan Keuangan Interim Yang Tidak Diaudit
No Nama Kegiatan
Usaha
Lokasi Tahun
Komersil
Status
Operasi
Jumlah Aset Satuan Mata
Uang
Persentase
(%)
1 PT Platinum
Surya Abadi
Sentosa
Real Estate Surabaya Non Operating 41.541 JUTAAN IDR 50.0
2 PT Tanrise
Makmur
Sentosa
R...


# Step 7: Instalasi Library Vector Database & Embedding
Kita menginstal `chromadb` untuk penyimpanan data vektor dan `langchain-huggingface` untuk mengubah teks menjadi angka koordinat (embedding).

In [3]:
!pip install -q chromadb sentence-transformers langchain-huggingface langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/7

# Step 8: Load Data Chunks
Alih-alih mengulang proses PDF yang berat, kita memuat kembali 9.744 potongan teks (chunks) yang sudah disimpan ke dalam file `final_chunks.json` pada Checkpoint 2.

In [ ]:
import json
import os

# Path menuju file JSON di repository github yang sudah di-clone
json_path = "RAG-Quantitative-Research-Engine-for-IDX/final_chunks.json"

if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        loaded_chunks = json.load(f)
    print(f"✅ Berhasil memuat {len(loaded_chunks)} chunks dari file JSON.")
else:
    print("❌ Error: File JSON tidak ditemukan. Periksa path repository Anda.")

✅ Berhasil memuat 9744 chunks dari file JSON.


# Step 9: Connecting to Persistent Vector Database
Kita memuat database vektor yang sudah tersimpan di repository GitHub.
Langkah ini sangat efisien karena kita tidak perlu melakukan proses embedding ulang terhadap 9.000+ chunks.

In [4]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os

# 1. Inisialisasi model embedding (harus sama dengan saat pembuatan database)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# 2. Tentukan path folder database di hasil clone github
DB_PATH = "RAG-Quantitative-Research-Engine-for-IDX/chroma_db_idx"

if os.path.exists(DB_PATH):
    vector_db = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space": "cosine"} # Pastikan metrik sesuai saat pembuatan
    )
    print("✅ Vector Database dari GitHub berhasil dimuat.")
else:
    print("❌ Error: Folder database tidak ditemukan di path tersebut.")

/tmp/ipykernel_1186/4070177153.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_1186/4070177153.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


✅ Vector Database dari GitHub berhasil dimuat.
